# 01 - Exploración de datos

Este notebook visualiza el manifiesto oficial. No modifica imágenes, etiquetas ni particiones.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd()
if not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
MANIFEST = ROOT / 'data' / 'processed' / 'dataset.csv'
if not MANIFEST.exists():
    raise FileNotFoundError('Ejecuta primero: frutas-data inventory, dedupe y split')
data = pd.read_csv(MANIFEST, dtype=str, keep_default_na=False)
print(f'{len(data):,} registros cargados desde {MANIFEST}')
data.head()

In [ ]:
summary = (data.groupby(['split', 'source', 'class_id'], as_index=False)
           .agg(images=('image_id', 'size'), original_labels=('original_label', 'nunique')))
summary

In [ ]:
train_counts = data[data['split'] == 'train']['class_id'].value_counts()
ax = train_counts.sort_values().plot.barh(figsize=(7, 4), title='Distribución binaria de entrenamiento')
ax.set_xlabel('Imágenes')
plt.tight_layout();

In [ ]:
eligible = data[(data['split'] == 'train') & (data['class_id'].isin(['fruta', 'verdura']))]
sample = eligible.sample(min(12, len(eligible)), random_state=42)
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for axis, (_, row) in zip(axes.flat, sample.iterrows()):
    axis.imshow(Image.open(row['path']).convert('RGB'))
    title = f"{row['class_id']} · {row['original_label']}"
    axis.set_title(title, fontsize=9)
    axis.axis('off')
for axis in axes.flat[len(sample):]:
    axis.axis('off')
plt.tight_layout();